In [ ]:
import logging
import warnings
from typing import TypeVar

import earthaccess
import fsspec
import xarray as xr
import zarr
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

from dask.distributed import Client, LocalCluster
from pyproj import Transformer

# EASE-Grid 2.0 Global (9 km) projection used by SMAP L4 x/y coordinates (meters).
EASE2_GLOBAL_EPSG = "EPSG:6933"

XarrayObj = TypeVar("XarrayObj", xr.Dataset, xr.DataArray)


def create_dask_cluster(
        n_workers: int = 4,
) -> tuple[Client, LocalCluster]:

    if "client" in locals() and "cluster" in locals():
        return (client, cluster) # type: ignore  # noqa: F821, F823
    else:
        print("Creating new local Dask client")
        cluster = LocalCluster(
            n_workers=n_workers,
            threads_per_worker=1,
            silence_logs=logging.ERROR)

        client = Client(cluster)
        return (client, cluster)


def silence_worker_warnings() -> None:
    warnings.filterwarnings("ignore")
    for name in ["distributed", "xarray", "py.warnings", "fsspec", "h5netcdf", "h5py"]:
        logging.getLogger(name).setLevel(logging.ERROR)


client, cluster = create_dask_cluster(n_workers=8)
client.run(silence_worker_warnings)

In [ ]:
refs = "https://its-live-data.s3-us-west-2.amazonaws.com/test-space/vds/SPL4SMGP.parquet"
daac_fs = earthaccess.get_fsspec_https_session()

fs = fsspec.filesystem(
    "reference",
    fo=refs,
    remote_protocol="https",
    asynchronous=True,
    remote_options={"asynchronous": True, **daac_fs.storage_options},
)

store = zarr.storage.FsspecStore(fs, read_only=True) # type: ignore
ds = xr.open_zarr(store, consolidated=False)
ds

In [ ]:
def latlon_bbox_to_ease(
    bbox: tuple[float, float, float, float],
) -> tuple[float, float, float, float]:
    """Convert a lat/lon bounding box to EASE-Grid 2.0 Global x/y bounds.

    The SMAP L4 x/y coordinates are in meters in the EASE-Grid 2.0
    Global projection (EPSG:6933), so a geographic bounding box must be
    reprojected before it can be used to index the grid.  EPSG:6933 is a
    cylindrical equal-area projection, so x depends only on longitude and
    y only on latitude; transforming the four corners and taking the
    min/max therefore yields exact axis-aligned bounds.

    Arguments:
        bbox: (west, south, east, north) in degrees (lon/lat, EPSG:4326).

    Returns:
        (x_min, y_min, x_max, y_max) in meters (EPSG:6933).
    """
    west, south, east, north = bbox
    transformer = Transformer.from_crs("EPSG:4326", EASE2_GLOBAL_EPSG, always_xy=True)
    xs, ys = transformer.transform([west, east, west, east], [south, south, north, north])
    return min(xs), min(ys), max(xs), max(ys)


def _bounds_slice(coord: xr.DataArray, lo: float, hi: float) -> slice:
    """Build a slice from lo to hi that respects a coordinate's order.

    xarray label slicing follows the coordinate's stored direction, and the
    SMAP L4 y coordinate is descending (north to south), so the slice bounds
    must be reversed for descending coordinates.
    """
    if float(coord[0]) > float(coord[-1]):
        return slice(hi, lo)
    return slice(lo, hi)


def _select_bbox(
    obj: XarrayObj,
    bbox: tuple[float, float, float, float],
) -> XarrayObj:
    """Select the x/y cells of a SMAP L4 grid falling inside a lat/lon box.

    Both the soil moisture and the land-model constants ride on the same
    EASE-Grid 2.0 cells, so selecting each with this shared helper guarantees
    their subsets carry identical x/y coordinates and align cell-for-cell.

    Raises:
        ValueError: If the bounding box does not overlap the dataset grid.
    """
    x_min, y_min, x_max, y_max = latlon_bbox_to_ease(bbox)
    subset = obj.sel(
        x=_bounds_slice(obj.x, x_min, x_max),
        y=_bounds_slice(obj.y, y_min, y_max),
    )
    if subset.sizes["x"] == 0 or subset.sizes["y"] == 0:
        msg = f"Bounding box {bbox} does not overlap the dataset grid."
        raise ValueError(msg)
    return subset


def sm_rootzone_timeseries(
    ds: xr.Dataset,
    bbox: tuple[float, float, float, float],
    freq: str = "3D",
    variable: str = "sm_rootzone",
    start: str | None = None,
    stop: str | None = None,
) -> xr.DataArray:
    """Spatially average a variable over a lat/lon box and aggregate in time.

    Arguments:
        ds (xr.Dataset): The SMAP L4 virtual dataset.
        bbox: (west, south, east, north) in degrees (lon/lat).
        freq (str): Pandas offset alias for the temporal aggregation window
            ("3D" = 3-day means).
        variable (str): The variable to aggregate.
        start (str | None): Optional start date (e.g. "2019" or "2019-01-01").
            If None, begins at the start of the dataset.
        stop (str | None): Optional end date (e.g. "2019" or "2019-12-31"),
            inclusive. If None, runs to the end of the dataset.

    Returns:
        xr.DataArray: A 1-D DataArray of the box-averaged, freq-aggregated variable
        indexed by time.

    Raises:
        ValueError: If the bounding box does not overlap the dataset grid.
    """
    subset = _select_bbox(ds, bbox)

    # Restrict to the requested date range before aggregating. xarray label
    # slicing is inclusive of both bounds and accepts partial dates, so passing
    # start="2019" and stop="2019" selects all of calendar year 2019.
    if start is not None or stop is not None:
        subset = subset.sel(time=slice(start, stop))

    # Mean over the box (NaN fill values over water/ice are skipped), then
    # aggregate the native 3-hourly steps into `freq` windows.
    spatial_mean = subset[variable].mean(dim=("x", "y"))
    return spatial_mean.resample(time=freq).mean()


bbox = (-111.0, 45.0, -106.0, 50.0)
sm_ts = sm_rootzone_timeseries(ds, bbox, start="2019", stop="2019")
sm_ts

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(sm_ts["time"], sm_ts.compute().data, color="darkblue", linewidth=1.5)
ax.set_ylabel("Soil Moisture [m3 m-3]")
ax.set_title("Rootzone Soil Moisture over Northern Great Plains \n45°–50°N, 106°–111°W")

# Let the tick locator/formatter adapt to whatever date range was requested,
# so a single-month window and a multi-year window both stay readable.
locator = mdates.AutoDateLocator()
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
